In [ ]:
-- ==========================================
-- Task 2.1 — Build and load the database
-- ==========================================
CREATE DATABASE IF NOT EXISTS noshowguard;
USE noshowguard;

CREATE TABLE clinics (
    clinic_id VARCHAR(5) PRIMARY KEY,
    clinic_name VARCHAR(40) NOT NULL,
    city VARCHAR(20) NOT NULL,
    tier VARCHAR(10),
    latitude DECIMAL(8,4),
    longitude DECIMAL(8,4),
    temp_c DECIMAL(5,1)
);

CREATE TABLE appointments (
    appointment_id VARCHAR(8) PRIMARY KEY,
    patient_id VARCHAR(8) NOT NULL,
    clinic_id VARCHAR(5) NOT NULL,
    booked_on DATE,
    appt_date DATE,
    age INT,
    gender VARCHAR(1),
    distance_km DECIMAL(5,1),
    fee_inr INT,
    booking_channel VARCHAR(10),
    visit_type VARCHAR(12),
    reminder_sent VARCHAR(3),
    prior_no_shows INT,
    status VARCHAR(10),
    lead_days INT,
    no_show INT,
    FOREIGN KEY (clinic_id) REFERENCES clinics(clinic_id)
);

-- Run load_data.sql generated in Part 1 here.

-- Check counts and no-shows
SELECT COUNT(*) AS total_appointments, SUM(no_show) AS total_no_shows FROM appointments;

-- Test duplicate insert error (produces "Duplicate entry 'A0001' for key 'PRIMARY'")
INSERT INTO appointments
SELECT * FROM appointments WHERE appointment_id = 'A0001';


-- ==========================================
-- Task 2.2 — No-show rate by clinic
-- ==========================================
-- With HAVING clause (at least 95 appointments)
SELECT
    c.clinic_name,
    c.city,
    COUNT(*) AS appointments,
    SUM(a.no_show) AS no_shows,
    ROUND(100.0 * SUM(a.no_show) / COUNT(*), 1) AS no_show_pct
FROM appointments a
JOIN clinics c ON a.clinic_id = c.clinic_id
GROUP BY c.clinic_id, c.clinic_name, c.city
HAVING COUNT(*) >= 95
ORDER BY no_show_pct DESC;

-- Without HAVING clause (all 8 clinics)
SELECT
    c.clinic_name,
    c.city,
    COUNT(*) AS appointments,
    SUM(a.no_show) AS no_shows,
    ROUND(100.0 * SUM(a.no_show) / COUNT(*), 1) AS no_show_pct
FROM appointments a
JOIN clinics c ON a.clinic_id = c.clinic_id
GROUP BY c.clinic_id, c.clinic_name, c.city
ORDER BY no_show_pct DESC;


-- ==========================================
-- Task 2.3 — Does booking far ahead matter?
-- ==========================================
SELECT
    CASE
        WHEN lead_days BETWEEN 1 AND 7 THEN '1-7'
        WHEN lead_days BETWEEN 8 AND 14 THEN '8-14'
        WHEN lead_days BETWEEN 15 AND 30 THEN '15-30'
        ELSE '31+'
    END AS lead_band,
    COUNT(*) AS appointments,
    SUM(no_show) AS no_shows,
    ROUND(100.0 * SUM(no_show) / COUNT(*), 1) AS no_show_pct
FROM appointments
GROUP BY lead_band
ORDER BY MIN(lead_days);


-- ==========================================
-- Task 2.4 — Month-by-month trend
-- ==========================================
WITH monthly AS (
    SELECT
        SUBSTR(appt_date, 1, 7) AS month,
        COUNT(*) AS appointments,
        ROUND(100.0 * SUM(no_show) / COUNT(*), 1) AS no_show_pct
    FROM appointments
    GROUP BY month
)
SELECT
    month,
    appointments,
    no_show_pct,
    ROUND(no_show_pct - LAG(no_show_pct, 1) OVER (ORDER BY month), 1) AS rate_change
FROM monthly
ORDER BY month;


-- ==========================================
-- Task 2.5 — Rank clinics inside each city
-- ==========================================
WITH clinic_rates AS (
    SELECT
        c.clinic_name AS clinic,
        c.city,
        ROUND(100.0 * SUM(a.no_show) / COUNT(*), 1) AS no_show_pct
    FROM appointments a
    JOIN clinics c ON a.clinic_id = c.clinic_id
    GROUP BY c.clinic_id, c.clinic_name, c.city
),
overall AS (
    SELECT ROUND(100.0 * SUM(no_show) / COUNT(*), 2) AS overall_pct FROM appointments
)
SELECT
    r.city,
    r.clinic,
    r.no_show_pct,
    RANK() OVER (PARTITION BY r.city ORDER BY r.no_show_pct DESC) AS rank_in_city,
    CASE
        WHEN r.no_show_pct > o.overall_pct THEN 'above'
        WHEN r.no_show_pct < o.overall_pct THEN 'below'
        ELSE 'equal'
    END AS vs_overall
FROM clinic_rates r
CROSS JOIN overall o
ORDER BY r.city, rank_in_city;